# TN1 — DS-TCN tầm nhìn 61, so 192 kênh với 64 kênh

## Cấu hình

Kiến trúc DS-TCN do nhóm tối ưu riêng cho bài toán này, **huấn luyện theo giao
thức TN1 hiện tại**.

| thành phần | giá trị |
|---|---|
| channels | **192** và **64** |
| số khối | 4, mỗi khối hai phép tích chập depthwise separable |
| kernel / độ giãn | 3 / 1, 2, 4, 8 |
| tầm nhìn | **61 mẫu** |
| chuẩn hoá | **không có** |
| dropout | `nn.Dropout(0.2)` |
| vào / ra | 200 → 25 |
| RevIN | tắt |
| tham số | **307.801** (c192) và **37.081** (c64) |

**Kiến trúc thì dựng lại nguyên bản, chế độ huấn luyện thì không.** Bản gốc dùng
loss hỗn hợp, learning rate và weight decay riêng. Ở đây giữ nguyên giao thức
TN1 — MSE, Adam lr 1e-4, weight decay 0, batch 64, 20 epoch, `corr` 0,9 — để so
được với các cấu hình TN1 khác. Vì vậy kết quả nói về **kiến trúc**, không tái
lập kết quả của bản gốc.

## Hai tuỳ chọn phải bổ sung vào code

Kiến trúc này khác `ds_tcn` của TN1 đúng hai điểm, cả hai đều cần cờ mới:

**`--norm none`.** Trước đây chỉ nhận `batch` hoặc `weight`. Kiến trúc này
**không có lớp chuẩn hoá nào** — đó chính là 3.072 tham số chênh lệch từng thấy,
bằng 8 lớp BatchNorm × 384.

**`--dropout_kind element`.** TN1 dùng `nn.Dropout1d`, xoá cả một kênh, theo
"spatial dropout" ở Bai mục 3.4. Kiến trúc này dùng `nn.Dropout` thường, xoá
từng phần tử. Hai loại chỉ khác nhau khi `dropout > 0`, mà mọi cấu hình TN1 để
0, nên chưa từng lộ ra.

## Đã đối chiếu với bản gốc, khớp từng bit

Dựng lại nguyên văn `CausalDSConv`, `TCNBlock`, `ForecastTCN`, chép trọng số
sang bản này rồi so:

```
tham số   gốc 307801   bản này 307801
cùng số tensor: True | cùng hình dạng: True
đầu ra giống hệt: True | chênh lớn nhất: 0.000e+00
```

Kiểm thêm bằng thực nghiệm: **đổi 139 mẫu đầu của cửa sổ không làm đổi đầu ra** —
đúng tầm nhìn 61.

## Tầm nhìn 61 — đọc kỹ chỗ này

Cửa sổ vào 200 mẫu, model chỉ thấy **61 mẫu gần nhất**, bỏ qua 70% đầu cửa sổ.

    TN1 ds_tcn      kernel 3, 6 khối  -> tầm nhìn 253, phủ trọn 200
    cấu hình này    kernel 3, 4 khối  -> tầm nhìn  61, thấy 61/200

Ở 50 Hz thì 61 mẫu là **1,2 giây**, trong khi một nhịp thở khoảng 4 giây. Model
không nhìn đủ một chu kỳ thở.

## Hai cấu hình chạy ở đây

| | channels | tham số |
|---|---:|---:|
| A | 192 | 307.801 |
| B | 64 | 37.081 |

B là đối chứng số kênh trên cùng kiến trúc, cùng tầm nhìn. Nếu A hơn B rõ thì
số kênh có tác dụng ở tầm nhìn này; nếu ngang nhau thì **một seed chưa đủ để nói
gì** — phải chạy thêm seed mới kết luận.

Đây là bước chọn nền trước khi khảo sát tầm nhìn, không phải kết luận về sức
chứa.

Ước lượng: A khoảng **2 giờ**, B khoảng **45 phút**.

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn. Hai cờ `--norm none` và `--dropout_kind element` chỉ có ở bản
mới, nên ô này phải chạy được thì mới đi tiếp.

**Ô này xoá `/content/UWB_RADAR`, tức xoá luôn `runs/` cục bộ.** Chạy lại nó
giữa chừng là mất kết quả chưa cất sang Drive. Mỗi fold `run_cv.py` tự nén sang
Drive nên không mất hẳn, nhưng phiên mới sẽ train lại từ fold đầu.

In [ ]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

## 2. Kiểm hai bản cài đặt

Bảy phép kiểm mỗi cấu hình. Ba phép nhắm đúng chỗ vừa bổ sung:

    mục 5   KHÔNG có lớp chuẩn hoá nào
    mục 6   dùng nn.Dropout chứ không phải nn.Dropout1d
    mục 7   in tầm nhìn 61 và cảnh báo mất 70% đầu cửa sổ

Số tham số phải ra đúng **307.801** và **37.081**.

Dùng `subprocess.run(check=True)` chứ không phải `!python`: lệnh `!` thất bại
thì notebook vẫn chạy tiếp, kể cả tới ô ngắt phiên. Kiểu này ném lỗi và dừng
hẳn, đúng ý nghĩa của một phép kiểm.

In [ ]:
import subprocess
subprocess.run("""python scripts/check_model.py --model ds_tcn --channels 192 --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element""".split(), check=True)

In [ ]:
import subprocess
subprocess.run("""python scripts/check_model.py --model ds_tcn --channels 64 --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element""".split(), check=True)

## 3. A — 192 kênh, 307.801 tham số

Tên cấu hình: `ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0`.

Hậu tố `_none` ghi cách chuẩn hoá, `_dpel` ghi loại dropout. Cả hai chỉ xuất
hiện khi khác mặc định nên tên các lần chạy trước không đổi — đã đối chiếu từng
ký tự cho cả 13 cấu hình đang có.

Khoảng **2 giờ**.

In [ ]:
import subprocess
subprocess.run("""python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 192 --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0""".split(), check=True)

## 4. B — 64 kênh, 37.081 tham số

Cùng kiến trúc, cùng tầm nhìn, chỉ đổi số kênh. Khoảng **45 phút**.

In [ ]:
import subprocess
subprocess.run("""python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 64 --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0""".split(), check=True)

## 5. Cất kết quả

In [ ]:
import subprocess
subprocess.run("""python scripts/save_results.py tn1 --out tn1_ds_tcn_rf61""".split(), check=True)

## 6. Bảng so

`compare_cv` chỉ thấy những gì có trong `runs/tn1/` của **phiên này**, tức hai
cấu hình vừa chạy. Ô clone đã xoá sạch `runs/` nên không có kết quả cũ ở đây.
Bảng đầy đủ dựng ở `TN1_final_evaluation.ipynb`, nơi gộp mọi tệp nén trên Drive.

Mốc từ TN1 để đặt cạnh:

| | tham số | tầm nhìn | cv_mean |
|---|---:|---:|---:|
| LSTM-352 | 1.502.713 | — | 0,7570 ± 0,0041 |
| LSTM-67 | 56.908 | — | 0,7532 ± 0,0020 |
| DS-TCN-64 | 56.281 | 253 | 0,7421 ± 0,0007 |

**Một seed thì đọc dè dặt.** Tám cấu hình TN1 có `seed_std` từ 0,0007 tới
0,0108, tức mỗi kiến trúc một khác — không có ngưỡng nhiễu chung áp cho mọi
dòng. Muốn nói hai cấu hình khác nhau thật thì phải chạy đủ seed cho chính
chúng, chứ không mượn `seed_std` của cấu hình khác.

In [ ]:
!python scripts/compare_cv.py --experiment tn1

## 7. Ngắt phiên

Kết quả đã nén sang Drive ở mục 5 nên ngắt ở đây không mất gì.

In [ ]:
from google.colab import runtime
runtime.unassign()